# SynopsisLake output review: all synopsis types

Reads the synopsis Parquet (`<folder>/syno/`) written by each task of the `SynopsisLake - Synopsis Overhead Exp` job (`-synopsisType` -1, 1, 3, 4, 6; 5 files x 10,000 records) from `/Volumes/cs598_ccc_research_7474657373242960/synopsis_lake/output` and displays it. The legacy `synout` folder is skipped.

In [0]:
# Synopsis Parquet written by each job task (all output folders except the legacy "synout")
OUTPUT_ROOT = "/Volumes/cs598_ccc_research_7474657373242960/synopsis_lake/output"

folders = sorted(
    f.name.rstrip("/") for f in dbutils.fs.ls(OUTPUT_ROOT)
    if f.isDir() and f.name.rstrip("/") != "synout"
)
syno_paths = {name: f"{OUTPUT_ROOT}/{name}/syno/" for name in folders}

for name, path in syno_paths.items():
    files = [f.name for f in dbutils.fs.ls(path)]
    print(f"{name}: {path}\n    {files}")

In [0]:
df_no_synopsis = spark.read.parquet(syno_paths["synout_no_synopsis"])
df_no_synopsis.printSchema()
print("Rows:", df_no_synopsis.count())
display(df_no_synopsis)

`no_synopsis` type is an identity operation, which does not return any meaningful value except the aggregated range region over the csv file partitions (10,000 records per partition)

In [0]:
df_count_hist_2d = spark.read.parquet(syno_paths["synout_count_hist_2d"])
df_count_hist_2d.printSchema()
print("Rows:", df_count_hist_2d.count())
display(df_count_hist_2d)

In [0]:
# Decode the 2D count histogram (type 1) into one heatmap per file.
# range = [minLon, maxLon, minLat, maxLat] (consistent with shape = cell width/height = extent / 128)
# counts[i][j]: i = lon bucket (dim 0), j = lat bucket (dim 1)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

rows = df_count_hist_2d.select("id", "range", "shape", "dimensionSizes", "counts", "totalNumOfItems").orderBy("id").collect()

fig, axes = plt.subplots(1, len(rows), figsize=(5 * len(rows), 4.5), squeeze=False)
for ax, r in zip(axes[0], rows):
    counts = np.array(r["counts"], dtype=float)  # shape (lon_buckets, lat_buckets)
    min_lon, max_lon, min_lat, max_lat = r["range"]
    nonzero = int((counts > 0).sum())
    im = ax.imshow(
        np.ma.masked_equal(counts.T, 0),  # transpose so lat is the y-axis; empty cells left blank
        origin="lower",
        extent=[min_lon, max_lon, min_lat, max_lat],
        aspect="auto",
        cmap="viridis",
        norm=LogNorm(vmin=1, vmax=max(counts.max(), 1)),
    )
    ax.set_title(
        f"file-{r['id']}: sum={counts.sum():,.0f} / {r['totalNumOfItems']:,}\n"
        f"non-empty cells {nonzero:,}/{counts.size:,}, max={counts.max():,.0f}",
        fontsize=9,
    )
    ax.set_xlabel("longitude")
    ax.set_ylabel("latitude")
    fig.colorbar(im, ax=ax, label="count (log)")

fig.suptitle("2D count histogram (128 x 128) per file")
plt.tight_layout()
plt.show()

# Per-file summary
for r in rows:
    c = np.array(r["counts"], dtype=float)
    print(f"file-{r['id']}: dims={r['dimensionSizes']}, cell={r['shape']}, range={r['range']}, "
          f"sum={c.sum():.0f}, totalNumOfItems={r['totalNumOfItems']}, non-empty={(c > 0).sum()}")

In [0]:
df_wavelet_2d = spark.read.parquet(syno_paths["synout_wavelet_2d"])
df_wavelet_2d.printSchema()
print("Rows:", df_wavelet_2d.count())
display(df_wavelet_2d)

**Incomplete Wavelet_2D**

TODO: "wavelet_2D" returned all zero in count and shape for csv_0, worth of investigating in the future (given the paper suggested wavelet is not there yet in the code)

In [0]:
df_geometric_hist = spark.read.parquet(syno_paths["synout_geometric_hist"])
df_geometric_hist.printSchema()
print("Rows:", df_geometric_hist.count())
display(df_geometric_hist)

In [0]:
# Geometric histogram (type 4): what corners / area / horizontal / vertical / shapes mean.
# Java path: DistributedBinaryData.buildFile (case 4) -> GeometricHistogramOpt.addRecord / fillCell.
# Each record is turned into a synthetic box [lon, lat, lon + 0.35, lat + 0.17] (cols 0-1 only).
# Approach: (1) check identities on the sums, (2) rebuild the grids independently in numpy and compare
# cell by cell, (3) plot the four grids for one file.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LON_PAD, LAT_PAD = 0.35, 0.17                     # synthetic box size used for type 4
NUM_FILE, NUM_RECORDS = 5, 10000
PER_FILE = NUM_RECORDS - 1                        # 1 id slot per chunk
GRIDS = ["corners", "area", "horizontal", "vertical"]

gh = spark.read.parquet(syno_paths["synout_geometric_hist"])
gh_rows = gh.orderBy("id").collect()

# Job chunks of 0.csv -> synthetic boxes, exactly as the Java code builds them
csv0 = pd.read_csv("/Volumes/cs598_ccc_research_7474657373242960/synopsis_lake/data/spatial_join/0.csv",
                   sep=";", header=None, names=["minLon", "minLat", "maxLon", "maxLat", "oid", "c5", "c6"])
csv0 = csv0.iloc[: NUM_FILE * PER_FILE].copy()
csv0["file_id"] = np.arange(len(csv0)) // PER_FILE


def rebuild_gh(boxes, min_lon, min_lat, lu, la, n_lon, n_lat):
    """Independent geometric histogram on a uniform grid.
    corners: # box corners in the cell; area: sum(intersection area / cell area);
    horizontal: sum(length of top/bottom edges inside the cell / cell width);
    vertical: sum(length of left/right edges inside the cell / cell height)."""
    C, A, H, V = (np.zeros((n_lon, n_lat)) for _ in range(4))
    x0, y0, x1, y1 = boxes.T
    i0 = np.clip(((x0 - min_lon) / lu).astype(int), 0, n_lon - 1)
    i1 = np.clip(((x1 - min_lon) / lu).astype(int), 0, n_lon - 1)
    j0 = np.clip(((y0 - min_lat) / la).astype(int), 0, n_lat - 1)
    j1 = np.clip(((y1 - min_lat) / la).astype(int), 0, n_lat - 1)
    for di in range(int((i1 - i0).max()) + 1):
        for dj in range(int((j1 - j0).max()) + 1):
            m = (i0 + di <= i1) & (j0 + dj <= j1)
            i, j = i0[m] + di, j0[m] + dj
            cx0, cx1 = min_lon + i * lu, min_lon + (i + 1) * lu
            cy0, cy1 = min_lat + j * la, min_lat + (j + 1) * la
            bx0, bx1, by0, by1 = x0[m], x1[m], y0[m], y1[m]
            ix = np.minimum(bx1, cx1) - np.maximum(bx0, cx0)   # overlap width
            iy = np.minimum(by1, cy1) - np.maximum(by0, cy0)   # overlap height
            np.add.at(A, (i, j), ix * iy / (lu * la))
            in_x = [(bx0 >= cx0) & (bx0 < cx1), (bx1 >= cx0) & ((bx1 < cx1) | (i == n_lon - 1))]
            in_y = [(by0 >= cy0) & (by0 < cy1), (by1 >= cy0) & ((by1 < cy1) | (j == n_lat - 1))]
            for ex in in_x:
                for ey in in_y:
                    np.add.at(C, (i, j), (ex & ey).astype(float))
            for ey in in_y:                                    # horizontal edges y = minLat, maxLat
                np.add.at(H, (i, j), np.where(ey, ix / lu, 0.0))
            for ex in in_x:                                    # vertical edges x = minLon, maxLon
                np.add.at(V, (i, j), np.where(ex, iy / la, 0.0))
    return {"corners": C, "area": A, "horizontal": H, "vertical": V}


summary, rebuilt = [], {}
for r in gh_rows:
    fid = r["id"]
    min_lon, max_lon, min_lat, max_lat = r["range"]        # [minLon, maxLon, minLat, maxLat]
    lu, la = r["shapes"]
    grids = {k: np.array(r[k], dtype=float) for k in GRIDS}
    n_lon, n_lat = grids["corners"].shape

    sub = csv0[csv0["file_id"] == fid]
    boxes = np.column_stack([sub["minLon"], sub["minLat"], sub["minLon"] + LON_PAD, sub["minLat"] + LAT_PAD])
    rb = rebuild_gh(boxes, min_lon, min_lat, lu, la, n_lon, n_lat)
    rebuilt[fid] = (grids, rb)
    # skip the last lon column when comparing: the zero-size phantom record (dataArray[0] = {0,0,0,0})
    # is clamped there by getBId
    diff = {k: np.abs(grids[k][:-1] - rb[k][:-1]).max() for k in GRIDS}
    n = len(sub)
    summary.append({
        "file": fid, "items": r["totalNumOfItems"], "grid": f"{n_lon}x{n_lat}",
        "shapes": [round(lu, 5), round(la, 5)],
        "range/128": [round((max_lon - min_lon) / n_lon, 5), round((max_lat - min_lat) / n_lat, 5)],
        "sum corners": grids["corners"].sum(), "expect 4N": 4 * n,
        "sum area": round(grids["area"].sum(), 2), "expect N*box/cell": round(n * LON_PAD * LAT_PAD / (lu * la), 2),
        "sum horizontal": round(grids["horizontal"].sum(), 2), "expect 2N*0.35/lu": round(2 * n * LON_PAD / lu, 2),
        "sum vertical": round(grids["vertical"].sum(), 2), "expect 2N*0.17/la": round(2 * n * LAT_PAD / la, 2),
        "non-empty cells": int((grids["area"] > 0).sum()),
        **{f"max |diff| {k}": round(v, 6) for k, v in diff.items()},
    })

summary_df = pd.DataFrame(summary)
print("Identities per file: shapes vs range/128, and grid sums vs closed-form expectations")
display(summary_df)

# --- Heatmaps for one file: stored grids (top) and the per-cell ratio meaning (bottom) ---
FID = 0
grids, rb = rebuilt[FID]
r0 = gh_rows[FID]
min_lon, max_lon, min_lat, max_lat = r0["range"]
lu, la = r0["shapes"]
extent = [min_lon, max_lon, min_lat, max_lat]
labels = {"corners": "# box corners in cell",
          "area": "sum(box-cell overlap area) / cell area",
          "horizontal": "sum(top+bottom edge length in cell) / cell width",
          "vertical": "sum(left+right edge length in cell) / cell height"}

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, k in zip(axes, GRIDS):
    g = grids[k].copy()
    g[-1, :] = 0                                           # hide phantom-record column
    im = ax.imshow(np.ma.masked_equal(g.T, 0), origin="lower", extent=extent, aspect="auto", cmap="viridis",
                   norm=plt.matplotlib.colors.LogNorm(vmin=max(g[g > 0].min(), 1e-3), vmax=g.max()))
    ax.set_title(f"file-{FID} {k}\n{labels[k]}", fontsize=9)
    ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle(f"Geometric histogram, file-{FID}: cell = {lu:.4f} deg x {la:.4f} deg (shapes), box = 0.35 x 0.17 deg")
plt.tight_layout()
plt.show()

# --- Per-cell relationships that show the meaning of each grid ---
m = (grids["corners"] > 0)
m[-1, :] = False
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].scatter(grids["corners"][m] / 4, grids["area"][m], s=4, alpha=0.5)
axes[0].plot([0, grids["corners"][m].max() / 4], [0, grids["corners"][m].max() / 4 * LON_PAD * LAT_PAD / (lu * la)], "r--",
             label="area if every box were fully inside its cell")
axes[0].set_xlabel("corners / 4  (~ boxes anchored in cell)"); axes[0].set_ylabel("area"); axes[0].legend(fontsize=7)
axes[1].scatter(grids["area"][m], grids["horizontal"][m], s=4, alpha=0.5)
axes[1].set_xlabel("area"); axes[1].set_ylabel("horizontal")
axes[2].scatter(grids["area"][m], grids["vertical"][m], s=4, alpha=0.5)
axes[2].set_xlabel("area"); axes[2].set_ylabel("vertical")
for ax in axes:
    ax.set_title(f"file-{FID}, non-empty cells", fontsize=9)
plt.tight_layout()
plt.show()

# Phantom record check (last lon column)
print("Last lon column (phantom record bucket) per grid, file-0:",
      {k: round(float(grids[k][-1].sum() - rb[k][-1].sum()), 4) for k in GRIDS})

**Geometric Histogram Discovery**

This summarizes the type 4 output (`synout_geometric_hist/syno/`, 5 rows, one per job file). The analysis is in cell 9. Java path: `DistributedBinaryData.buildFile` (case 4), then `GeometricHistogramOpt.addRecord` / `fillCell`.

**Three different rectangles; do not mix them up**

| Term | What it is | Stored as |
| --- | --- | --- |
| `range` | Extent of the whole file: min/max over all of its boxes | `[minLon, maxLon, minLat, maxLat]` (note the order: lon pair first, then lat pair) |
| **cell** | One bucket of the 128 x 128 grid laid over `range`. Cell `[i][j]` covers lon `[minLon + i*w, minLon + (i+1)*w)` and lat `[minLat + j*h, minLat + (j+1)*h)` | Implicit: array index `[i][j]`, with i = lon bucket and j = lat bucket |
| object box | One record. Type 4 ignores the real box in CSV cols 0-3 and builds a synthetic box from cols 0-1 | `[lon, lat, lon + 0.35, lat + 0.17]` (not stored; it is only in the `.bin` file) |

`shapes` = `[w, h]` = cell width and height in degrees = `(range extent) / 128`. This matched exactly in all 5 files (e.g. file-0: `[1.12502, 0.61909]`). It is the same quantity as `shape` in the count histogram.

**What each grid stores per cell** (all four are 128 x 128, indexed `[lon bucket][lat bucket]`)

| Column | Per-cell value | Units | Check on the sums (N = 9,999 boxes per file) |
| --- | --- | --- | --- |
| `corners` | number of box corners inside the cell | count | 39,998 = 4N + 2 (the +2 comes from the phantom record below) |
| `area` | sum over boxes of (box-cell overlap area / cell area) | cell areas | equals N x (0.35 x 0.17) / (w x h) exactly, e.g. 854.20 for file-0 |
| `horizontal` | sum of top + bottom edge length inside the cell / cell width | cell widths | 2N x 0.35 / w, minus about 63.4 from the phantom record |
| `vertical` | sum of left + right edge length inside the cell / cell height | cell heights | equals 2N x 0.17 / h exactly, e.g. 5,491.38 for file-0 |

* An independent numpy rebuild of all four grids from the CSV matches the stored Parquet **cell by cell** (max abs difference 0) in all 5 files. The only exception is the last lon column, which holds the phantom record.
* Because every value is normalized by cell size, `shapes` is needed to convert back to degrees, and the same `[i][j]` covers different ground in different files (each file has its own `range`).
* Only 980-1,291 of 16,384 cells are non-empty per file. A 0.35 x 0.17 box is smaller than a cell (about 1.1 x 0.6 deg), so each box touches 1-4 cells.

**Worked example: file-3, cell `[0][46]`**

These values are not query regions. Every box in these grids is a data object from the input CSV, one per record. A query window only appears at estimate time, when the same four grids are built for it.

The cell is 1.1185 deg x 0.5793 deg (`shapes`), so its area is 0.648 sq deg. One synthetic box (0.35 x 0.17 deg, area 0.0595 sq deg) crosses the cell's right edge, and 0.147 deg of its width lies inside the cell:

| Column | Value | Where it comes from |
| --- | --- | --- |
| `corners` | 2 | the box's two left corners are inside the cell; its two right corners fall in the next cell |
| `area` | 0.0385 | the overlap is 0.147 x 0.17 = 0.0249 sq deg, and 0.0249 / 0.648 = 0.0385 |
| `horizontal` | 0.262 | the top and bottom edges each run 0.147 deg inside the cell: 2 x 0.147 / 1.1185 = 0.262 |
| `vertical` | 0.293 | only the left edge is inside, and it is 0.17 deg long: 0.17 / 0.5793 = 0.293 |

The same box puts its other 2 corners, and the remaining 0.203 deg of each horizontal edge, into the neighbouring cell. So `horizontal` and `vertical` are **not** edge counts. They are total edge **lengths**, measured in cell widths or cell heights, which is why they are fractions here.

**Intuitive reading of each value**
* `corners`: a plain count of box corners in the cell. It is the only true count among the four.
* `area`: the expected number of boxes covering a random point in the cell. 0.0385 means about a 4% chance that a random point in this cell lies inside some box.
* `horizontal`: the expected number of horizontal box edges that a random vertical line across the cell would cross.
* `vertical`: the expected number of vertical box edges that a random horizontal line across the cell would cross.

So these are not just finer-grained counts. Together they are what is needed to estimate how many boxes intersect a query window.

**Why these four numbers** (An et al. geometric histogram, paper Table 1: used for `SpatialJoinEst`)
Two rectangles intersect when a corner of one is inside the other, or when edges cross. For a query window Q, `GeometricHistogramOpt.selectivityEstimation` builds the same four grids for Q (C', A', H', V') and estimates the number of intersecting boxes as:

sum over cells of (C x A' + A x C' + H x V' + V x H') / 4

* C x A': box corners that fall inside Q
* A x C': Q corners that fall inside boxes
* H x V' and V x H': horizontal edges crossing vertical edges

**Issues found**
* **Phantom record:** the loader loop starts at `i = 1`, so `dataArray[0]` stays `{0,0,0,0}` but is still added. `getBId` clamps it into the last lon column, which adds 2 corners and about -63.4 to `horizontal` in every file.
* **Synthetic boxes:** real object boxes are about 0.002 deg wide, but the histogram uses 0.35 x 0.17 deg boxes, which are over 100 times larger. Estimates therefore describe the synthetic polygons, not the real data (see the 538 estimate vs 590 synthetic vs 432 real comparison in the `geometric_histogram` notebook).
* **No shared grid:** every file has its own `range` and therefore its own cell boundaries. The per-file histograms cannot be added cell by cell; they must be aligned first (the paper's Align-Reshape-Merge problem).


In [0]:
df_uniform_samples = spark.read.parquet(syno_paths["synout_uniform_samples"])
df_uniform_samples.printSchema()
print("Rows:", df_uniform_samples.count())
display(df_uniform_samples)

# --- Drill-down: where does each sampleData point come from? ---
# Java: DistributedBinaryData.buildFile collects (lon, lat) = CSV cols 0-1 for every record of the chunk,
# then UniformSamples.Sample does Collections.shuffle(items) and keeps the first ceil(N * sampleRatio).
import math
import numpy as np
import pandas as pd

PER_FILE = 9999  # NUM_RECORDS=10000 minus the chunk-id slot
csv0_us = pd.read_csv("/Volumes/cs598_ccc_research_7474657373242960/synopsis_lake/data/spatial_join/0.csv",
                      sep=";", header=None, names=["minLon", "minLat", "maxLon", "maxLat", "oid", "c5", "c6"])
csv0_us["csv_row"] = csv0_us.index
csv0_us["chunk"] = csv0_us.index // PER_FILE

matches = []
for r in df_uniform_samples.orderBy("id").collect():
    n_expected = math.ceil(r["totalNumOfItems"] * r["sampleRatio"])
    for k, (lon, lat) in enumerate(r["sampleData"]):
        hit = csv0_us[(csv0_us["minLon"] == lon) & (csv0_us["minLat"] == lat)]
        matches.append({"file": r["id"], "sample_idx": k, "lon": lon, "lat": lat,
                        "expected_n": n_expected, "stored_n": len(r["sampleData"]),
                        "csv_row": hit["csv_row"].tolist(), "csv_chunk": hit["chunk"].tolist(),
                        "row_in_chunk": (hit["csv_row"] - r["id"] * PER_FILE).tolist(),
                        "c5": hit["c5"].tolist()})
matches_df = pd.DataFrame(matches)
display(matches_df)

print("every sample matches exactly one CSV row:", (matches_df["csv_row"].str.len() == 1).all())
print("every sample comes from its own file's chunk:",
      all(c == [f] for c, f in zip(matches_df["csv_chunk"], matches_df["file"])))
print("samples are the chunk's first rows:",
      sorted(sum(matches_df["row_in_chunk"].tolist(), [])) [:6] == list(range(6)))
print("first two CSV rows (cols 0-1):", csv0_us.iloc[:2, :2].values.tolist())

**Uniform Samples**

This summarizes the type 6 output (`synout_uniform_samples/syno/`, 5 rows, one per job file). The analysis is in cell 12. Java path: `DistributedBinaryData.buildFile` (case 6), then `samples/UniformSamples.Sample`.

**Columns**

| Column | Meaning | Value in this run |
| --- | --- | --- |
| `range` | Extent of the file, `[minLon, maxLon, minLat, maxLat]` (same order as the histograms) | e.g. file-0 `[-179.68, -36.02, -61.07, 18.00]` |
| `sampleData` | List of sampled points; each inner array is one point `[lon, lat]` | 6 points per file |
| `sampleRatio` | Fraction of the file's records that were kept | 0.0006 |
| `totalNumOfItems` | Records in the file | 9,999 |
| `rawFilePath` / `type` | Path to the file's `.bin` data, and the synopsis type | `file-<id>.bin` / 6 |

There is no grid, so there are no cells and no `shape(s)` column.

**How the sample is built**
* The rate is hard-coded by resolution in `SynopsisOverheadExp`: 128 gives 0.0006, 256 gives 0.001, 512 gives 0.005.
* `buildFile` collects `(lon, lat)` = CSV cols 0-1 for every record of the chunk. `Sample` then calls `Collections.shuffle(items)` and keeps the first `ceil(N x ratio)` = `ceil(9,999 x 0.0006)` = **6** points.

**Drill-down: is `sampleData` just the first CSV rows? No.**
* All 30 stored points (5 files x 6) match **exactly one** CSV row, and each comes from its own file's chunk. For example, file-0's samples are CSV rows 9985, 5941, 2212, 6420, 4980 and 2872.
* They are spread across the whole chunk, not taken from the top. The first two CSV rows, `(-80.028, -2.851)` and `(-55.404, 5.742)`, are not in any sample.
* Each point is the object's `(minLon, minLat)` corner only. CSV cols 2-3 (the max corner) are dropped, so the sample stores points, not boxes.
* The sample follows the data mix: file-0's 6 points include one c5=0 point (lon -159.8, Pacific) alongside c5=1 and c5=2 points.

**Differences from the histograms**
* **No phantom record:** the sample list is built only from real records (loop from `i = 1`), so the all-zero `{0,0}` record never appears here. (The geometric histogram adds that record to its last lon column; the count histogram drops it because (0, 0) lies outside its range.)
* **Not reproducible:** `Collections.shuffle` is called without a seed, so every job run stores a different set of points.
* **Tiny at this scale:** 6 points x 2 doubles = 96 bytes per file, compared with 128 x 128 doubles (about 131 KB) for the count histogram and 4 such grids for the geometric histogram. At the paper's scale (1,000,000 records per file) the same rate gives 600 points.

**How it would be used**
* Estimated number of objects in a query window = (sample points inside the window) / `sampleRatio`. Each point here stands for 1 / 0.0006 = about **1,667** objects.
* With only 6 points per file, estimates jump in steps of about 1,667. A small window (such as the 1 x 1 deg Misiones window in the `geometric_histogram` notebook, with 432 real objects) will usually get an estimate of 0. Samples only become useful at larger record counts or higher rates.
* Samples merge easily: files with the same rate can simply be concatenated, unlike the histograms, which need their grids aligned first.


In [0]:
# Load ALL input CSVs (spatial_join/0.csv .. 5.csv) and count rows per category (column 5) in each.
# (The job itself currently reads only 0.csv; SynopsisOverheadExp.loadData chunks it in order,
#  each chunk = 1 id slot + (NUM_RECORDS - 1) records, so job file-k = rows k*9999 .. (k+1)*9999 - 1 of 0.csv.)

#* column 5 of the original csv remained mostly mysterious, trying to plot what are these in the distribution and heatmap sense.

import glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = "/Volumes/cs598_ccc_research_7474657373242960/synopsis_lake/data/spatial_join"
COLS = ["minLon", "minLat", "maxLon", "maxLat", "oid", "c5", "c6"]
NUM_FILE, NUM_RECORDS = 5, 10000
records_per_file = NUM_RECORDS - 1

csv_paths = sorted(glob.glob(f"{DATA_DIR}/*.csv"), key=lambda p: int(os.path.basename(p).split(".")[0]))
raw = pd.concat(
    [pd.read_csv(p, sep=";", header=None, names=COLS).assign(csv_id=int(os.path.basename(p).split(".")[0]))
     for p in csv_paths],
    ignore_index=True,
)
print(f"Loaded {len(csv_paths)} CSVs, {len(raw):,} rows")

# Per-CSV extent and column 6 values
extent = raw.groupby("csv_id").agg(rows=("oid", "size"),
                                   minLon=("minLon", "min"), maxLon=("maxLon", "max"),
                                   minLat=("minLat", "min"), maxLat=("maxLat", "max"),
                                   c5_values=("c5", lambda s: sorted(s.unique())),
                                   c6_values=("c6", lambda s: sorted(s.unique())),
                                   oid_unique=("oid", lambda s: s.is_unique))
display(extent.round(2).reset_index())

cat_counts = raw.pivot_table(index="csv_id", columns="c5", values="oid", aggfunc="size", fill_value=0)
cat_counts.columns = [f"c5={c}" for c in cat_counts.columns]
cat_counts["total"] = cat_counts.sum(axis=1)

# Overall share for comparison
overall_share = raw["c5"].value_counts(normalize=True).sort_index()
print("Category share across all CSVs:", {f"c5={k}": f"{v:.1%}" for k, v in overall_share.items()})

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_df = cat_counts.drop(columns="total")
plot_df.plot(kind="bar", ax=axes[0], rot=0)
axes[0].set_title("Rows per category (column 5) per CSV")
axes[0].set_xlabel("csv id")
axes[0].set_ylabel("rows")
for cont in axes[0].containers:
    axes[0].bar_label(cont, fontsize=6)

(plot_df.div(plot_df.sum(axis=1), axis=0) * 100).plot(kind="bar", stacked=True, ax=axes[1], rot=0)
axes[1].set_title("Category share per CSV (%)")
axes[1].set_xlabel("csv id")
axes[1].set_ylabel("% of CSV rows")
axes[1].legend(loc="lower right", fontsize=7)
plt.tight_layout()
plt.show()

display(cat_counts.reset_index())

# Job chunks of 0.csv (what the current job actually reads)
job = raw[raw["csv_id"] == 0].reset_index(drop=True).iloc[: NUM_FILE * records_per_file].copy()
job["file_id"] = job.index // records_per_file
job_counts = job.pivot_table(index="file_id", columns="c5", values="oid", aggfunc="size", fill_value=0)
job_counts.columns = [f"c5={c}" for c in job_counts.columns]
print("Job file chunks of 0.csv (rows per c5):")
display(job_counts.reset_index())

# --- Map entities as points per CSV, colored by c5 (same colors as the bar charts: C0 blue, C1 orange, C2 green, ...) ---
# Boxes are tiny, so (minLon, minLat) stands in for the whole box; check the span first.
raw["lon_span"] = raw["maxLon"] - raw["minLon"]
raw["lat_span"] = raw["maxLat"] - raw["minLat"]
print(f"Box span (deg): lon median={raw['lon_span'].median():.4f}, max={raw['lon_span'].max():.4f}; "
      f"lat median={raw['lat_span'].median():.4f}, max={raw['lat_span'].max():.4f}; "
      f"rows with span >= 1 deg: {((raw['lon_span'] >= 1) | (raw['lat_span'] >= 1)).sum():,}")

cats = sorted(raw["c5"].unique())
color_map = {c: f"C{i}" for i, c in enumerate(cats)}  # c5=0 blue, 1 orange, 2 green, 3 red, 4 purple, 5 brown

csv_ids = sorted(raw["csv_id"].unique())
fig, axes = plt.subplots(2, 3, figsize=(18, 9), squeeze=False, sharex=True, sharey=True)
for ax, cid in zip(axes.flat, csv_ids):
    g = raw[raw["csv_id"] == cid]
    # draw the common categories first so rare categories stay visible on top
    for c in sorted(g["c5"].unique(), key=lambda c: -(g["c5"] == c).sum()):
        pts = g[g["c5"] == c]
        ax.scatter(pts["minLon"], pts["minLat"], s=0.5, alpha=0.5, color=color_map[c],
                   label=f"c5={c} ({len(pts):,})", rasterized=True)
    ax.set_title(f"{cid}.csv: {len(g):,} rows", fontsize=9)
    ax.set_xlabel("longitude")
    ax.set_ylabel("latitude")
    ax.legend(markerscale=10, fontsize=7, loc="lower left")
fig.suptitle("Entities as points (minLon, minLat) per CSV, colored by c5")
plt.tight_layout()
plt.show()

# --- Hypothesis: c5 = 60-degree longitude band index, floor((lon + 180) / 60) ---
# 0: [-180,-120), 1: [-120,-60), 2: [-60,0), 3: [0,60), 4: [60,120), 5: [120,180]
print(raw.groupby("c5")[["minLon", "maxLon"]].agg(["min", "max"]).round(2))
for name, lon in {"minLon": raw["minLon"], "maxLon": raw["maxLon"],
                  "box center": (raw["minLon"] + raw["maxLon"]) / 2}.items():
    pred = np.floor((lon + 180) / 60).astype(int)
    print(f"c5 == floor(({name} + 180) / 60): {(pred == raw['c5']).mean():.5%} match, "
          f"{(pred != raw['c5']).sum()} mismatches")

for ax in axes.flat:
    for b in (-120, -60, 0, 60, 120):
        ax.axvline(b, color="k", ls="--", lw=0.8)
fig.suptitle("Entities per CSV, colored by c5 (dashed: 60-degree longitude band edges)")
display(fig)

# Mismatches per CSV (box center rule) and whether oids repeat across CSVs
pred_center = np.floor(((raw["minLon"] + raw["maxLon"]) / 2 + 180) / 60).astype(int)
print("Mismatches (box center rule) per CSV:", (pred_center != raw["c5"]).groupby(raw["csv_id"]).sum().to_dict())
oid_csvs = raw.groupby("oid")["csv_id"].nunique()
print(f"oids appearing in more than one CSV: {(oid_csvs > 1).sum():,} of {len(oid_csvs):,} distinct oids")

**Discovery of Geospatial distribution of the queries**

This covers all six input CSVs in `/Volumes/cs598_ccc_research_7474657373242960/synopsis_lake/data/spatial_join/` (1,380,350 rows; analysis in cell 9). Note that these rows are the **data objects** that get summarized, not range queries; the query workload is a separate dataset (see README).

**Row format** `minLon;minLat;maxLon;maxLat;oid;c5;c6`
* Cols 0-3 are the bounding box of each object. Boxes are tiny (median 0.0024 deg x 0.0017 deg). Only 382 rows span 1 deg or more (max 28 x 12 deg), so treating an object as its `(minLon, minLat)` corner is fine for maps.
* Col 4 `oid` is an object id and is nearly unique: 1,380,256 distinct values, with 34 ids repeated across CSVs.
* Col 5 `c5` is the **60-degree longitude band index** `floor((lon + 180) / 60)`: 0 = [-180,-120), 1 = [-120,-60), 2 = [-60,0), 3 = [0,60), 4 = [60,120), 5 = [120,180]. This holds for 99.98% of rows; the 237-330 mismatches (depending on which lon is used) are boxes that straddle a band edge. It is a plain meridian cut, not land/ocean or country (e.g. Paraguay is split between c5=1 and 2).
* Col 6 `c6` is always 1 in every CSV, so it carries no information here.

**The six CSVs form an overlapping 3 x 2 spatial partition**

| CSV | Rows | Lon | Lat | c5 bands | Region |
| --- | --- | --- | --- | --- | --- |
| 0 | 97,750 | -180 to -36 | -90 to 19 | 0, 1, 2 | South America, south/east Pacific |
| 1 | 141,956 | -84 to 83 | -90 to 18 | 1, 2, 3, 4 | Africa, south Atlantic, west Indian Ocean |
| 2 | 240,644 | 36 to 179 | -78 to 20 | 3, 4, 5 | South/SE Asia, Australia, Oceania |
| 3 | 300,000 | -179 to -36 | -18 to 70 | 0, 1, 2 | North/Central America |
| 4 | 300,000 | -84 to 85 | -18 to 81 | 1, 2, 3, 4 | Europe, Middle East, north Africa |
| 5 | 300,000 | 36 to 179 | -18 to 72 | 3, 4, 5 | Asia |

* There are three longitude groups with hard edges at -84, -36 and 36 deg, and two latitude groups (south and north) split near -18 to 20 deg. Neighbouring partitions overlap by about 40-48 deg of longitude and about 36 deg of latitude, so the partition MBRs overlap; partitions are not disjoint tiles.
* The northern CSVs are each capped at exactly 300,000 rows, which suggests sampling or truncation. The southern ones are smaller and uneven (98K-241K).
* Overall c5 share: 7.5% / 21.7% / 12.8% / 26.8% / 8.9% / 22.2% for bands 0-5. Data is densest in Europe/Africa (band 3) and East Asia/Oceania (band 5).

**What the current job sees**
* The job reads only `0.csv`, and only its first 49,995 rows (5 files x 9,999 records), which is 3.6% of all rows. Each job file is a consecutive slice, not a spatial partition, so all 5 files cover nearly the same South American extent.
* `0.csv` is only partly shuffled: the c5 mix differs by slice (c5=1 is 29-49%, and c5=0 is mostly in files 0-2). Merging per-file synopses therefore sees overlapping extents with different internal skew, which is the Align/Reshape scenario from the paper.
* Every per-file synopsis grid spans about 140 deg x 54-108 deg, so 128 x 128 cells are about 1.1 deg x 0.6-0.8 deg and only about 5-6% of cells are non-empty.

**Next steps**
* Run the job per CSV (or with spatial chunks) to get synopses whose extents match real partitions, then compare merged estimates across the overlap regions.
* Use the query workload from the README link to check estimate error against these distributions.